## Importing dataframes

In [270]:
import pandas as pd
import re
import numpy as np

In [271]:
df_fiction = pd.read_csv('/home/gabriel/Documents/Lovecraft-project/data/bronze/lovecraft_fiction.csv')
df_collab = pd.read_csv('/home/gabriel/Documents/Lovecraft-project/data/bronze/lovecraft_collaborations.csv')
df_magazine = pd.read_csv('/home/gabriel/Documents/Lovecraft-project/data/bronze/lovecraft_magazines.csv')


## Standardizing the dataframes format

In order to have a better organization and to facilitate manipulations within the dataframes, put everything in lower case and get rid of any whitespaces that may occur. 


In [272]:
# put columns title in lower case

df_fiction.columns = df_fiction.columns.str.lower()
df_collab.columns = df_collab.columns.str.lower()
df_magazine.columns = df_magazine.columns.str.lower()

# create lists of column items for each dataframe

cols_fiction = [col for col in df_fiction.columns]
cols_collab = [col for col in df_collab.columns]
cols_magazine = [col for col in df_magazine.columns]

# put columns items in lower case and strip whitespaces

df_fiction[cols_fiction] = df_fiction[cols_fiction].apply(lambda x: x.str.strip().str.lower() if x.dtype == "object" else x)
df_collab[cols_collab] = df_collab[cols_collab].apply(lambda x: x.str.strip().str.lower() if x.dtype == "object" else x)
df_magazine[cols_magazine] = df_magazine[cols_magazine].apply(lambda x: x.str.strip().str.lower() if x.dtype == "object" else x)

# further formating that will be convenient later

df_fiction = df_fiction.drop(columns=['id'])
df_fiction = df_fiction.rename(columns={'date written': 'date_written', 'date published': 'date_published'})
df_collab = df_collab.rename(columns={'date written': 'date_written', 'date published': 'date_published'})
df_magazine_with_year = df_magazine.rename(columns={'year': 'year_published'})
df_magazine_with_year = df_magazine_with_year.drop(columns=['month'])
df_magazine = df_magazine.drop(columns=['month','year'])



In [273]:
df_magazine_with_year

,year_published,title,magazine
0,1916,the alchemist,the united amateur
1,1917,a reminiscence of dr. samuel johnson,the united amateur
2,1918,the beast in the cave,the vagrant
3,1919,memory,the united co-operative
4,1919,beyond the wall of sleep,pine cones
...,...,...,...
100,1959,the mysterious ship,the shuttered room and other pieces
101,1959,old bugs,the shuttered room and other pieces
102,1994,discarded draft of the shadow over innsmouth,the shadow over innsmouth
103,1994,the hoard of the wizard-beast,the hoard of the wizard-beast and one other


## Merging dataframes

Now we take the dataframe with H.P. Lovecrat's sole publications and the dataframe with collaborations and concatenate them to obtain all of his publications. We create two new columns for year of writing and year of publication so we can work on more reliable data, since the precise data (such as month) is uncertain. 

In [274]:
df_collab_fiction = pd.concat([df_fiction, df_collab], ignore_index=True)
df_collab_fiction['year_written'] = df_collab_fiction['date_written'].str.extract(r'(\d{4})').astype('Int64')
df_collab_fiction['year_published'] = df_collab_fiction['date_published'].str.extract(r'(\d{4})').astype('Int64')

We now verify which titles are do not, so we can refine our data frame.

In [275]:
non_intersection = set(df_collab_fiction['title']).symmetric_difference(set(df_magazine['title']))
print('tales that do not match:\n')
for title in non_intersection:
    print(title)    


tales that do not match:

the little glass bottle
the secret cave or john lees adventure
discarded draft of the shadow over innsmouth
under the pyramids
the horror at martin's beach
medusa's coil
pickman’s model
the black lotus
vine terror
deaf, dumb and blind
the beast in the cave
the salem horror
herbert west—reanimator
deaf dumb and blind
till a'the seas
medusa’s coil
the red brain
the sorcery of aphlar
bothon
the horror at martin’s beach
something from above
herbert west–reanimator
four o'clock
the mystery of the grave-yard
the werewolf of ponkert
imprisoned with the pharaohs
“till a’ the seas”
the mysterious ship
satan's servants
pickman's model


To correct this, we create this function to normalize the titles considering the following differences that appeared. 

In [276]:
def normalize_title(title):
    if pd.isna(title):
        return None
    
    title = title.lower()
    
    # normalize unicode quotes/dashes
    title = (
        title.replace("’", "'")
             .replace("‘", "'")
             .replace("“", "")
             .replace("”", "")
             .replace("–", "-")
             .replace("—", "-")
    )
    
    # remove subtitles ("or ...")
    title = re.split(r"\s+or\s+", title)[0]
    
    # remove punctuation
    title = re.sub(r"[^\w\s]", "", title)
    
    # normalize spaces
    title = re.sub(r"\s+", "_", title.strip())
    
    return title

# applying 

df_collab_fiction["title"] = df_collab_fiction["title"].apply(normalize_title)
df_magazine["title"] = df_magazine["title"].apply(normalize_title)
df_magazine_with_year["title"] = df_magazine_with_year["title"].apply(normalize_title)


In [277]:
df = pd.merge(df_collab_fiction, df_magazine, on="title", how="left").sort_values(by="year_published", ascending=True)
df

,title,date_written,date_published,form,collaborators,year_written,year_published,magazine
0,the_alchemist,1908,nov 1916,short story,NaN,1908,1916,the united amateur
3,a_reminiscence_of_dr_samuel_johnson,sum-early fall 1917,sep 1917,short story,NaN,1917,1917,the united amateur
2,dagon,jul 1917,nov 1919,short story,NaN,1917,1919,the vagrant
5,beyond_the_wall_of_sleep,spr 1919,oct 1919,short story,NaN,1919,1919,pine cones
9,the_white_ship,c. oct 1919,nov 1919,short story,NaN,1919,1919,the united amateur
...,...,...,...,...,...,...,...,...
8,the_transition_of_juan_romero,16 september 1919,1944,short story,NaN,1919,1944,marginalia
66,bothon,1930,1946,NaN,henry s. whitehead,1930,1946,NaN
98,satans_servants,1935,1949,NaN,robert bloch,1935,1949,NaN
75,four_oclock,1922,1949,NaN,sonia greene,1922,1949,NaN


We check now which titles are only in "df_magazine" to include them into "df". 

In [278]:
# creates a set of the titles that are in the magazine dataframe but not in the main dataframe, and prints them out

titles_to_be_add = set(df_magazine['title']) - set(df['title'])
print('these are the titles that should be added to the main dataframe:\n')
for title in titles_to_be_add:
    print(title)

these are the titles that should be added to the main dataframe:

the_little_glass_bottle
under_the_pyramids
the_mystery_of_the_graveyard
the_secret_cave
the_mysterious_ship
till_a_the_seas
the_beast_in_the_cave
discarded_draft_of_the_shadow_over_innsmouth


Now we create a new magazine data frame to merge with df

In [279]:
df_magazine_new = df_magazine_with_year[df_magazine_with_year['title'].isin(titles_to_be_add)]

df = pd.concat([df, df_magazine_new], ignore_index=True).sort_values(by="year_published", ignore_index=True)
df

,title,date_written,date_published,form,collaborators,year_written,year_published,magazine
0,the_alchemist,1908,nov 1916,short story,NaN,1908,1916,the united amateur
1,a_reminiscence_of_dr_samuel_johnson,sum-early fall 1917,sep 1917,short story,NaN,1917,1917,the united amateur
2,the_beast_in_the_cave,NaN,NaN,NaN,NaN,<NA>,1918,the vagrant
3,dagon,jul 1917,nov 1919,short story,NaN,1917,1919,the vagrant
4,the_white_ship,c. oct 1919,nov 1919,short story,NaN,1919,1919,the united amateur
...,...,...,...,...,...,...,...,...
112,the_little_glass_bottle,NaN,NaN,NaN,NaN,<NA>,1959,the shuttered room and other pieces
113,the_secret_cave,NaN,NaN,NaN,NaN,<NA>,1959,the shuttered room and other pieces
114,the_mystery_of_the_graveyard,NaN,NaN,NaN,NaN,<NA>,1959,the shuttered room and other pieces
115,the_mysterious_ship,NaN,NaN,NaN,NaN,<NA>,1959,the shuttered room and other pieces


With a research on ChatGPT using the Deep Research tool, we identify the following


| Original Key                         | Canonical Title                                      | Year (written/published)   | Authorship/Notes                                                      | Source(s)                                                      |
|-------------------------------------|------------------------------------------------------|----------------------------|------------------------------------------------------------------------|----------------------------------------------------------------|
| **the_werewolf_of_ponkert**         | *The Werewolf of Ponkert*                            | 1925                       | Story by H. Warner Munn (Lovecraft suggested the idea, but not author) (~reliable) | Wikipedia (Munn)                                                |
| **satans_servants**                 | *Satan's Servants*                                   | written 1935; pub. 1949    | Robert Bloch (Lovecraft revised the original draft) (collaboration/revision) | Lovecraft Wiki                                                 |
| **the_sorcery_of_aphlar**           | *The Sorcery of Aphlar*                              | Dec 1934                   | Duane W. Rimel (Lovecraft revised) (originally credited to Rimel)      | Lovecraft Wiki                                                 |
| **four_oclock**                     | *Four O'Clock*                                       | written 1922; pub. 1949    | Sonia H. Greene (idea from Lovecraft) (Lovecraft only provided inspiration) | Lovecraft Wiki                                                 |
| **the_salem_horror**                | *The Salem Horror*                                   | pub. May 1937              | Henry Kuttner (primary author; independently written)                  | Lovecraft Wiki                                                 |
| **something_from_above**            | *Something from Above*                               | Dec 1930                   | Donald Wandrei (purely authored by Wandrei)                            | ISFDB / Weird Tales                                            |
| **the_little_glass_bottle**         | *The Little Glass Bottle*                            | written 1897 (Juvenilia)   | H. P. Lovecraft (written as a child)                                   | HPLovecraft.com                                                |
| **the_red_brain**                   | *The Red Brain*                                      | written 1927; pub. 1927    | Donald Wandrei (Lovecraft encouraged him to write it)                  | Project Gutenberg                                              |
| **imprisoned_with_the_pharaohs**    | *Imprisoned with the Pharaohs*                       | written Feb 1924; pub. 1924| H. P. Lovecraft & Harry Houdini (Lovecraft ghostwrote for Houdini)     | Wikipedia                                                      |
| **bothon**                          | *Bothon*                                             | written 1930; pub. 1946    | Henry S. Whitehead (Lovecraft may have contributed ideas; final authorship uncertain) | Lovecraft Wiki                                                 |
| **the_beast_in_the_cave**           | *The Beast in the Cave*                              | written 1904/05; pub. 1918 | H. P. Lovecraft (juvenilia; first version written at age 13)           | Wikipedia                                                      |
| **vine_terror**                     | *Vine Terror*                                        | Sep 1934                   | Howard Wandrei (not Lovecraft; pulp horror story)                      | Wikisource (Weird Tales 1934)                                  |
| **the_secret_cave**                 | *The Secret Cave; or, John Lees’s Adventure*         | written 1898 (Juvenilia)   | H. P. Lovecraft (written at age 8)                                     | Lovecraft Wiki                                                 |
| **the_mystery_of_the_graveyard**    | *The Mystery of the Graveyard; or, A Dead Man's Revenge* | written 1898 (Juvenilia) | H. P. Lovecraft (written at age 8)                                     | Readers Library PDF                                            |
| **discarded_draft_of_the_shadow_over_innsmouth** | *Discarded Draft of “The Shadow Over Innsmouth”* | 1931 (draft); pub. 2005 | H. P. Lovecraft (unpublished early draft; fragment)                    | Wikipedia (anthology)                                          |
| **under_the_pyramids**              | *Imprisoned with the Pharaohs*                       | Feb 1924                   | (same as “Imprisoned with the Pharaohs”: Lovecraft & Houdini)          | Wikipedia                                                      |
| **the_black_lotus**                 | *The Black Lotus*                                    | written 1934; pub. 1935    | Robert Bloch (pulp horror; not Lovecraft; similar to “Servants”)       | Wikipedia bibliography                                         |
| **the_mysterious_ship**             | *The Mysterious Ship*                                | written 1902 (unfinished)  | H. P. Lovecraft (juvenilia, unfinished)                                | Lovecraft Wiki                                                 |

---

### Additional Notes

- *“The Werewolf of Ponkert”* is not by H. P. Lovecraft, but by H. Warner Munn, inspired by Lovecraft (high confidence).
- *“Bothon”* has uncertain Lovecraft involvement: S. T. Joshi suggests Lovecraft only contributed plot ideas and synopsis; the final text is by Whitehead (medium confidence).
- *“Satan’s Servants”* and *“The Black Lotus”* are by Robert Bloch, with Lovecraft acting as mentor/revisor (high confidence).
- All juvenilia (1890–1905) are identified with high confidence by HPLovecraft.com and Lovecraft wikis.
- Cross-confirmation comes from the Joshi/Schultz bibliography and Arkham House publications (notably posthumous collections from 1959).

---

### Sources

Main sources include:

- HPLovecraft.com (official archives)
- Lovecraft Wiki (lovecraft.fandom.com)
- S. T. Joshi / David E. Schultz bibliographies (via Wikipedia)
- Original magazine records (*Weird Tales*, etc.)

Each table entry corresponds to these references. In uncertain cases, a confidence note is provided, but the table reflects the current academic consensus on Lovecraft’s bibliography.

Therefore, there are some titles which are duplicates, drafts, unfinished stories or did not have the involvement of H.P Lovecraft at all. We remove those

In [280]:
titles_to_remove = [
'till_athe_seas',
'imprisoned_with_the_pharaohs',
'discarded_draft_of_the_shadow_over_innsmouth',
'the_mysterious_ship',
'the_salem_horror',
'something_from_above',
'bothon',
'vine_terror',
'the_black_lotus'    
]

df = df[~df['title'].isin(titles_to_remove)]

df

,title,date_written,date_published,form,collaborators,year_written,year_published,magazine
0,the_alchemist,1908,nov 1916,short story,NaN,1908,1916,the united amateur
1,a_reminiscence_of_dr_samuel_johnson,sum-early fall 1917,sep 1917,short story,NaN,1917,1917,the united amateur
2,the_beast_in_the_cave,NaN,NaN,NaN,NaN,<NA>,1918,the vagrant
3,dagon,jul 1917,nov 1919,short story,NaN,1917,1919,the vagrant
4,the_white_ship,c. oct 1919,nov 1919,short story,NaN,1919,1919,the united amateur
...,...,...,...,...,...,...,...,...
110,four_oclock,1922,1949,NaN,sonia greene,1922,1949,NaN
111,old_bugs,c. jul 1919,1959,short story,NaN,1919,1959,the shuttered room and other pieces
112,the_little_glass_bottle,NaN,NaN,NaN,NaN,<NA>,1959,the shuttered room and other pieces
113,the_secret_cave,NaN,NaN,NaN,NaN,<NA>,1959,the shuttered room and other pieces


Now, since we no longer need "date_written" and "date_published" columns, we may remove them 

In [281]:
df = df.drop(columns=['date_written', 'date_published'])
df

,title,form,collaborators,year_written,year_published,magazine
0,the_alchemist,short story,NaN,1908,1916,the united amateur
1,a_reminiscence_of_dr_samuel_johnson,short story,NaN,1917,1917,the united amateur
2,the_beast_in_the_cave,NaN,NaN,<NA>,1918,the vagrant
3,dagon,short story,NaN,1917,1919,the vagrant
4,the_white_ship,short story,NaN,1919,1919,the united amateur
...,...,...,...,...,...,...
110,four_oclock,NaN,sonia greene,1922,1949,NaN
111,old_bugs,short story,NaN,1919,1959,the shuttered room and other pieces
112,the_little_glass_bottle,NaN,NaN,<NA>,1959,the shuttered room and other pieces
113,the_secret_cave,NaN,NaN,<NA>,1959,the shuttered room and other pieces


In [282]:
df.to_csv('/home/gabriel/Documents/Lovecraft-project/data/bronze/lovecraft_works.csv', index=False)